In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Jaser12614@"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load the Grazioso Salvare logo
with open("Grazioso Salvare Logo.png", "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode()

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    html.Div([
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image),
                style={
                    'height': '120px',
                    'display': 'block',
                    'margin': 'auto'
                }
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),
        html.H1(
            "Grazioso Salvare Rescue Animal Dashboard",
            style={
                'textAlign': 'center',
                'marginTop': '20px'
            }
        ),
        html.H3(
            "Developed by Tanvi Chowdhury",
            style={
                'textAlign': 'center',
                'color': '#555'
            }
        )
    ]),
    html.Hr(),
        # Rescue Type Filters
    html.Div([
        html.H3("Filter by Rescue Type"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All Animals', 'value': 'Reset'},
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'}
            ],
            value='Reset',
            inline=True
        )
    ], style={
        'textAlign': 'center',
        'padding': '20px'
    }),
    html.Hr(),
    # Interactive animal data table
    dash_table.DataTable(
        id='datatable-id',
        # Display columns from MongoDB
        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],
        
        # Load animal records
        data=df.to_dict('records'),
        
        # Allow only one animal to be selected
        row_selectable='single',
        selected_rows=[0],

        # Enable pagination
        page_size=10,
        page_action='native',

        # Enable sorting and filtering
        sort_action='native',
        filter_action='native',

        # Improve table appearance
        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'padding': '10px',
            'minWidth': '120px',
            'maxWidth': '200px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        },

        style_header={
            'backgroundColor': '#f2f2f2',
            'fontWeight': 'bold'
        }
    ),
        html.Br(),
        html.Hr(),
    # Display the breed chart and geolocation map side by side
    html.Div(
        children=[
            # Breed distribution chart
            html.Div(
                id='graph-id',
                style={
                    'width': '50%',
                    'minWidth': '0'
                }
            ),
            
            # Geolocation map
            html.Div(
                id='map-id',
                style={
                    'width': '50%',
                    'minWidth': '0',
                    'minHeight': '500px'
                }
            )
        ],
        style={
            'display': 'flex',
            'width': '100%',
            'gap': '20px',
            'alignItems': 'flex-start'
        }
    )
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    if filter_type == 'Water':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'Labrador Retriever|Chesapeake Bay Retriever|Newfoundland',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'Mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'Disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    else:
        query = {}
    # Retrieve matching records from MongoDB
    filtered_data = pd.DataFrame.from_records(db.read(query))
    if '_id' in filtered_data.columns:
        filtered_data.drop(columns=['_id'], inplace=True)
    return filtered_data.to_dict('records'), [0]

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    # Get the currently displayed data
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame(viewData)

    if dff.empty:
        return html.P("No animal data available.")

    # Count the 10 most common breeds
    breed_counts = dff['breed'].value_counts().head(10)

    # Create a pie chart
    fig = px.pie(
        names=breed_counts.index,
        values=breed_counts.values,
        title="Top 10 Animal Breeds"
    )

    fig.update_layout(
        height=450,
        margin=dict(l=20, r=20, t=60, b=20)
    )

    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        selected_columns = []
        
    return [
        {
            'if': {'column_id': column},
            'background_color': '#D2F3FF'
        }
        for column in selected_columns
    ]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, selected_rows):

    # Get the currently displayed data
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame(viewData)

    if dff.empty:
        return html.P("No animals found for this filter.")

    # Use the selected row or default to the first row
    if selected_rows and selected_rows[0] < len(dff):
        row = selected_rows[0]
    else:
        row = 0

    animal = dff.iloc[row]

    # Get the animal's location
    lat = pd.to_numeric(animal['location_lat'], errors='coerce')
    lon = pd.to_numeric(animal['location_long'], errors='coerce')

    if pd.isna(lat) or pd.isna(lon):
        return html.P("Location unavailable for this animal.")

    return [
        dl.Map(
            style={
                'width': '100%', 
                'height': '500px'
            },
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(),

                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(animal['breed'])),

                        dl.Popup([
                            html.H3("Animal Information"),
                            html.P(f"Name: {animal['name']}"),
                            html.P(f"Breed: {animal['breed']}")
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://greenclark-tripodnadia-3000.codio.io/proxy/8050/
